# 🤖 Notebook 03: Modelos Base con PySpark MLlib

**Proyecto:** Predicción de duración de casos judiciales cerrados en Bolivia  
**Autores:** Vergara & Patiño  
**Materia:** Tecnologías Emergentes (TI26)  

Este notebook implementa dos modelos base de regresión usando PySpark MLlib:
1. **Regresión Lineal**
2. **Random Forest Regressor**

Se utiliza el CSV procesado generado en el notebook 02.

In [ ]:
# Instalar dependencias
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.mllib.evaluation import RegressionMetrics
import time

spark = SparkSession.builder.master("local[*]").appName("ModelosBase").getOrCreate()
print("Spark iniciado correctamente ✅")

## 1. Carga de Datos Procesados

In [ ]:
# Cargar datos procesados
# NOTA: Asegúrate de que 'casos_procesados.csv' esté disponible.
# Ruta local del archivo.
df = spark.read.csv("../data/processed/casos_procesados.csv", header=True, inferSchema=True)

print(f"Registros cargados: {df.count()}")
print(f"Columnas: {df.columns}")
df.printSchema()

## 2. División Train/Test

In [ ]:
# Dividir en train (70%) y test (30%)
train, test = df.randomSplit([0.7, 0.3], seed=42)

print(f"Train: {train.count()} registros")
print(f"Test:  {test.count()} registros")

## 3. Definición del Pipeline de Features

In [ ]:
# Columnas
numeric_features = [
    "victima_edad_num", "denunciado_edad_num", 
    "hecho_gestion_num", "hecho_mes_num", 
    "hecho_dia_num", "hecho_dia_semana_num"
]
categorical_feature = "delito"

# Etapas del pipeline
imputer = Imputer(
    inputCols=numeric_features, 
    outputCols=numeric_features
).setStrategy("mean")

assembler_num = VectorAssembler(
    inputCols=numeric_features, 
    outputCol="num_vector"
)

scaler = StandardScaler(
    inputCol="num_vector", 
    outputCol="scaled_num_vector", 
    withStd=True, withMean=True
)

indexer = StringIndexer(
    inputCol=categorical_feature, 
    outputCol="cat_index", 
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCol="cat_index", 
    outputCol="cat_vec"
)

final_assembler = VectorAssembler(
    inputCols=["scaled_num_vector", "cat_vec"], 
    outputCol="features"
)

print("Pipeline de features definido ✅")
print(f"  Features numéricas: {numeric_features}")
print(f"  Feature categórica: {categorical_feature}")

## 4. Modelo 1: Regresión Lineal

In [ ]:
# Regresión Lineal
lr = LinearRegression(
    featuresCol="features", 
    labelCol="duracion_dias"
)

pipeline_lr = Pipeline(stages=[
    imputer, assembler_num, scaler, indexer, encoder, final_assembler, lr
])

print("Entrenando Regresión Lineal...")
start = time.time()
model_lr = pipeline_lr.fit(train)
time_lr = time.time() - start
print(f"Entrenamiento completado en {time_lr:.2f} segundos ✅")

In [ ]:
# Evaluación Regresión Lineal
pred_lr = model_lr.transform(test)
pred_rdd_lr = pred_lr.select("prediction", "duracion_dias").rdd.map(
    lambda r: (float(r[0]), float(r[1]))
)
metrics_lr = RegressionMetrics(pred_rdd_lr)

print("="*50)
print("📊 RESULTADOS - REGRESIÓN LINEAL")
print("="*50)
print(f"  RMSE:  {metrics_lr.rootMeanSquaredError:.2f}")
print(f"  MAE:   {metrics_lr.meanAbsoluteError:.2f}")
print(f"  R²:    {metrics_lr.r2:.4f}")
print(f"  Tiempo: {time_lr:.2f}s")
print("="*50)

## 5. Modelo 2: Random Forest Regressor

In [ ]:
# Random Forest
rf = RandomForestRegressor(
    featuresCol="features", 
    labelCol="duracion_dias", 
    numTrees=50, 
    seed=42
)

pipeline_rf = Pipeline(stages=[
    imputer, assembler_num, scaler, indexer, encoder, final_assembler, rf
])

print("Entrenando Random Forest (50 árboles)...")
start = time.time()
model_rf = pipeline_rf.fit(train)
time_rf = time.time() - start
print(f"Entrenamiento completado en {time_rf:.2f} segundos ✅")

In [ ]:
# Evaluación Random Forest
pred_rf = model_rf.transform(test)
pred_rdd_rf = pred_rf.select("prediction", "duracion_dias").rdd.map(
    lambda r: (float(r[0]), float(r[1]))
)
metrics_rf = RegressionMetrics(pred_rdd_rf)

print("="*50)
print("🌲 RESULTADOS - RANDOM FOREST")
print("="*50)
print(f"  RMSE:  {metrics_rf.rootMeanSquaredError:.2f}")
print(f"  MAE:   {metrics_rf.meanAbsoluteError:.2f}")
print(f"  R²:    {metrics_rf.r2:.4f}")
print(f"  Tiempo: {time_rf:.2f}s")
print("="*50)

## 6. Comparación de Modelos Base

In [ ]:
import pandas as pd

resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Random Forest'],
    'RMSE': [metrics_lr.rootMeanSquaredError, metrics_rf.rootMeanSquaredError],
    'MAE': [metrics_lr.meanAbsoluteError, metrics_rf.meanAbsoluteError],
    'R²': [metrics_lr.r2, metrics_rf.r2],
    'Tiempo (s)': [time_lr, time_rf]
})

print("\n📋 TABLA COMPARATIVA - MODELOS BASE")
print("="*60)
print(resultados.to_string(index=False))
print("="*60)

# Guardar resultados parciales
resultados.to_csv("../results/resultados_modelos_base.csv", index=False)
print("\n✅ Resultados guardados en 'resultados_modelos_base.csv'")

In [ ]:
# Cerrar sesión Spark
spark.stop()
print("\n🏁 Spark detenido. Modelos base completados exitosamente.")
print("   Continúa con el notebook 04 para modelos avanzados (XGBoost, LightGBM, Stacking).")